# Come RTR spiega una singola istanza

Questo notebook mostra, passo per passo, come viene spiegato il punteggio di un documento in RuleTreeRank. Il punteggio finale è la somma di due contributi:

    f(x) = r(x) + s(x)

- r(x) è il punteggio grezzo del primo stadio: un albero poco profondo manda il documento in una foglia e gli assegna la media delle etichette di quella foglia.
- s(x) è la correzione del secondo stadio: dentro la foglia si cercano i documenti più vicini (kNN) usando una distanza appresa, e si fa la media dei loro residui.

Alleniamo il modello con RuleCard come modello di distanza e spieghiamo una istanza reale: quale foglia, quali vicini, con quali regole, e come si arriva al punteggio finale. Alla fine confrontiamo i vicini scelti da RuleCard con quelli scelti dal PDT.

Usiamo di proposito RuleTreeRank base su dati sintetici, con foglie abbastanza grandi: così la distanza appresa viene davvero usata dal kNN e la spiegazione è significativa.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "ruletreerank").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
from RuleTree import RuleTreeRegressor
from ltr_utility import ModelParam
from ruletreerank import KNNRegFast, RuleTreeRank, RuleCardPairwiseDistance, PairwiseDistanceTree

# dati sintetici con una relazione lineare nota più un po' di rumore
rng = np.random.default_rng(0)
n, d = 300, 6
X = rng.normal(size=(n, d))
w = np.array([1.5, -0.8, 0.6, 0.0, 0.3, -0.4])
y = X @ w + rng.normal(0, 0.1, n)
q = np.repeat(np.arange(30), 10)  # 30 query da 10 documenti
cols = [f"f{j}" for j in range(d)]

def costruisci_rtr(distance_f):
    return RuleTreeRank(
        distance_f=distance_f,
        aggregation_f=ModelParam(KNNRegFast, {"n_neighbors": 5, "n_jobs": 1}),
        base_regressor=RuleTreeRegressor(max_depth=2, random_state=0),
        dist_objective="dist",
    )

rtr_rc = costruisci_rtr(ModelParam(RuleCardPairwiseDistance, {
    "base_regressor": ModelParam(RuleTreeRegressor, {"max_depth": 2, "random_state": 0}),
    "feature_diff": True, "feature_concat": False, "feature_sq_diff": False,
    "subsample": 0.8, "learning_rate": 0.2, "max_n_iter": 30, "patience": 3,
}))
rtr_rc.fit(X, y, q)
print("modello allenato")

C:\Users\manzo\anaconda3\envs\rtr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


modello allenato


## Primo stadio: il punteggio grezzo r(x)

Scegliamo una istanza da spiegare e guardiamo in quale foglia finisce e che punteggio grezzo riceve.

In [2]:
i = 7  # istanza da spiegare
xi = X[i:i+1]
qi = q[i:i+1]

r = rtr_rc.predict(xi, q=qi, output="score")[0]
foglia = rtr_rc._shallow_dt.apply(xi)[0]

print("istanza:", np.round(xi[0], 3))
print("foglia raggiunta:", foglia, "  (la sigla codifica il percorso di decisione nell'albero)")
print(f"punteggio grezzo r(x) = {r:.4f}  (media delle etichette della foglia)")

istanza: [ 1.346  0.781  0.264 -0.314  1.458  1.96 ]
foglia raggiunta: Rrr   (la sigla codifica il percorso di decisione nell'albero)
punteggio grezzo r(x) = 2.4036  (media delle etichette della foglia)


## Secondo stadio: la correzione s(x)

Dentro la foglia si cercano i cinque documenti di training più vicini all'istanza, usando la distanza appresa da RuleCard. La correzione è la media dei loro residui (residuo = etichetta vera meno punteggio grezzo).

In [3]:
agg = rtr_rc._leaf_dist_map[foglia]
dist_model = agg.custom_metric_func
print("documenti di training nella foglia:", agg._fit_X.shape[0])

idx = agg.knn_neighbors_fast(xi).astype(int)[0]
residui_vicini = agg._y[idx].ravel()
distanze = dist_model.predict(np.repeat(xi, len(idx), axis=0), agg._fit_X[idx])

print("\nvicini scelti (indice nella foglia | distanza appresa | residuo):")
for k in range(len(idx)):
    print(f"  vicino {idx[k]:>3}   dist={distanze[k]:.4f}   residuo={residui_vicini[k]:+.4f}")

s = rtr_rc.predict(xi, q=qi, output="corr")[0]
print(f"\ncorrezione s(x) = {s:.4f}")
print(f"media dei residui dei vicini = {residui_vicini.mean():.4f}  (coincide con s per costruzione)")

documenti di training nella foglia: 42

vicini scelti (indice nella foglia | distanza appresa | residuo):
  vicino   1   dist=0.0000   residuo=-1.2071
  vicino  14   dist=5.1814   residuo=-0.4979
  vicino   2   dist=5.9809   residuo=-0.8442
  vicino   5   dist=6.7903   residuo=+0.1520
  vicino  29   dist=7.0478   residuo=+1.3743

correzione s(x) = -0.2046
media dei residui dei vicini = -0.2046  (coincide con s per costruzione)


## Perché quei vicini: le regole di RuleCard

La distanza tra due documenti è una somma di voci additive, una per ogni round di boosting. Ogni voce guarda la differenza assoluta di una feature tra i due documenti. Le prime voci sono le più importanti, perché a ogni round si sceglie la feature che riduce di più l'errore rimasto.

In [4]:
regole = dist_model.get_rules(columns_names=cols)
print(f"numero di voci additive: {len(regole)}  (fallback attivo: {dist_model._fallback})\n")
print("prime voci della scheda (ognuna è un alberello su una feature di differenza):")
for v in regole[:5]:
    print("  su", v["features"], "->", v["rules"])

numero di voci additive: 30  (fallback attivo: False)

prime voci della scheda (ognuna è un alberello su una feature di differenza):
  su ['diff(f3)'] -> {'node_id': 'R', 'is_leaf': False, 'prediction': 0.42826008807355387, 'prediction_probability': 5.690441799842561, 'log_odds': nan, 'prediction_classes_': array([-8.44950272, -8.2540763 , -7.83246256, -7.77072855, -7.14986302,
       -6.68777419, -6.45092593, -6.21507211, -6.14209503, -5.96661385,
       -5.86163946, -5.56028902, -5.0798273 , -5.01249495, -4.83598317,
       -4.57309462, -4.50027411, -4.30758155, -4.30637991, -4.17398069,
       -4.1536859 , -4.09755086, -3.67881729, -3.63492835, -3.61186415,
       -3.41343819, -3.38782565, -3.38592859, -3.31615251, -3.19458156,
       -3.15825483, -3.14193993, -2.67466661, -2.42509369, -2.29801005,
       -2.18471032, -1.98385212, -1.90973267, -1.77563882, -1.63990217,
       -1.63254073, -1.58240069, -1.3310408 , -1.32346842, -1.2681052 ,
       -1.24089576, -1.21363331, -1.0285850

## Il punteggio finale

Sommando i due contributi si ottiene f(x), e ordinando i documenti della query per f(x) decrescente si ottiene la posizione dell'istanza nel ranking.

In [5]:
f = rtr_rc.predict(xi, q=qi, output="full")[0]
print(f"f(x) = r(x) + s(x) = {r:.4f} + ({s:.4f}) = {r + s:.4f}")
print(f"f(x) restituito dal modello = {f:.4f}")

# posizione nel ranking della sua query
mask_q = q == qi[0]
f_query = rtr_rc.predict(X[mask_q], q=q[mask_q], output="full")
ordine = np.argsort(-f_query)
pos = int(np.where(ordine == np.where(np.flatnonzero(mask_q) == i)[0][0])[0][0]) + 1
print(f"posizione dell'istanza nella sua query: {pos} su {mask_q.sum()}")

f(x) = r(x) + s(x) = 2.4036 + (-0.2046) = 2.1991
f(x) restituito dal modello = 2.1991


posizione dell'istanza nella sua query: 1 su 10


## Confronto con il PDT

Alleniamo lo stesso RTR ma con il PDT come modello di distanza e guardiamo la stessa istanza. I vicini e le distanze cambiano, perché la distanza è appresa in modo diverso: il PDT usa un solo albero, RuleCard una somma di voci ordinate per importanza.

In [6]:
rtr_pdt = costruisci_rtr(ModelParam(PairwiseDistanceTree, {
    "base_regressor": ModelParam(RuleTreeRegressor, {"max_depth": 2, "random_state": 0}),
    "feature_diff": True, "feature_concat": False, "feature_sq_diff": False,
    "subsample": 0.8,
}))
rtr_pdt.fit(X, y, q)

agg_p = rtr_pdt._leaf_dist_map[rtr_pdt._shallow_dt.apply(xi)[0]]
idx_p = agg_p.knn_neighbors_fast(xi).astype(int)[0]
dist_p = agg_p.custom_metric_func.predict(np.repeat(xi, len(idx_p), axis=0), agg_p._fit_X[idx_p])

print("vicini scelti da RuleCard:", idx.tolist())
print("vicini scelti dal PDT    :", idx_p.tolist())
print("\ndistanze RuleCard:", np.round(distanze, 3).tolist())
print("distanze PDT     :", np.round(dist_p, 3).tolist())
print("\nStessa istanza, distanza appresa diversa: il PDT spiega con un albero,")
print("RuleCard con una somma di voci ordinate per importanza.")

vicini scelti da RuleCard: [1, 14, 2, 5, 29]
vicini scelti dal PDT    : [0, 1, 2, 3, 4]

distanze RuleCard: [0.0, 5.181, 5.981, 6.79, 7.048]
distanze PDT     : [8.201, 8.201, 8.201, 8.201, 8.201]

Stessa istanza, distanza appresa diversa: il PDT spiega con un albero,
RuleCard con una somma di voci ordinate per importanza.


## In sintesi

La spiegazione di una istanza si legge così: il primo stadio dice in quale gruppo (foglia) finisce e con quale punteggio di partenza; il secondo stadio mostra quali documenti simili l'hanno corretta, di quanto, e in base a quali differenze di feature. Con RuleCard la parte di distanza è una scheda a punti leggibile voce per voce, con il PDT è un percorso in un albero.